# **VERSION 1 OF THE XGBOOST CODE, WORK IN PROGRESS**

# **SKELETON OF THE CODE and MISSING DATASETS**

If we decide on using the LSTM and flight landing datasets and combining them by timestamps for predictions:

Step 1 — load LSTM predictions and flight landing dataset

Step 2 — merge both datasets by timestamp

Step 3 — extract labels (landed/did not land) from the flight dataset and connect X and y

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

#load datasets
# lstm_predictions = pd.read_csv("lstm_predictions.csv") #placeholder name

lstm_predictions = None  #<-- replace this

# flight_df = pd.read_csv("flight_landing_data.csv") #placeholder name

flight_df = None  #<-- replace this


#if we merge datasets by a shared datetime column so that each row represents one flight approach

LABEL_COLUMN = "landed"  # <-- replace with actual column name
TIMESTAMP_COLUMN = "timestamp"  # <-- replace with actual column name

lstm_predictions[TIMESTAMP_COLUMN] = pd.to_datetime(lstm_predictions[TIMESTAMP_COLUMN])
flight_df[TIMESTAMP_COLUMN] = pd.to_datetime(flight_df[TIMESTAMP_COLUMN])

combined_df = pd.merge(lstm_predictions, flight_df, on=TIMESTAMP_COLUMN, how="inner")


#extract labels
# 0 = did not land (unsafe), 1 = landed successfully (safe)

y = combined_df[LABEL_COLUMN]
X = combined_df.drop(columns=[LABEL_COLUMN, TIMESTAMP_COLUMN])


#train test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    #keeps class balance in both splits
)


#train XGBoost classifier

model = xgb.XGBClassifier(
    n_estimators=100,       #number of trees TUNE THIS!!
    max_depth=4,            #depth of each tree TUNE THIS!!
    learning_rate=0.1,      #step size TUNE THIS!!!
    subsample=0.8,          #fraction of samples per tree
    colsample_bytree=0.8,   #fraction of features per tree
    eval_metric="logloss",
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)


#evaluation

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Did not land", "Landed"]))

#confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Did not land", "Landed"],
            yticklabels=["Did not land", "Landed"])
plt.title("Confusion Matrix — XGBoost Landing Safety")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()


#feature importance
#shows which features (air quality vs flight conditions) matter most

xgb.plot_importance(model, importance_type="gain", title="Feature Importance")
plt.tight_layout()
plt.show()

TypeError: 'NoneType' object is not subscriptable